# Try the automation scripts with a fake `claude`
Modules *CLI scripting* and *Local pipelines*, exercises. Using Claude Code needs a subscription or an API key, and this notebook uses none. Here a small program called `claude` always answers with a canned JSON response. The course scripts (`gate.sh`, `migra.sh`) run **for real**: you see the exit codes, the marker files and the retry of failed files. Only the AI is fake.

Run the cells from top to bottom (Shift+Enter on each).

In [ ]:
%%bash
which jq >/dev/null || (apt-get -qq update && apt-get -qq install -y jq >/dev/null)
mkdir -p /tmp/finto /tmp/lab
jq --version

In [ ]:
%%writefile /tmp/finto/claude
#!/usr/bin/env bash
# "claude" FINTO, solo per questo notebook: non chiama nessuna IA.
# Legge stdin e restituisce una risposta JSON preconfezionata, come farebbe `claude -p --output-format json`.
[ -t 0 ] || cat > /dev/null
[ "${CLAUDE_FINTO_FALLISCE:-0}" = 1 ] && exit 1
case "$*" in
  *Migrate*)
     # per la migrazione: risponde OK, tranne per i file con "rotto" nel nome
     case "$*" in *rotto*) echo "FAIL";; *) echo "OK";; esac ;;
  *)
     echo '{"is_error":false,"permission_denials":[],"total_cost_usd":0.01,"structured_output":{"bloccanti":[{"file":"app.py","riga":3,"problema":"divisione per zero","scenario":"con b = 0 la funzione solleva ZeroDivisionError"}]}}' ;;
esac

In [ ]:
import os
os.chmod('/tmp/finto/claude', 0o755)
os.environ['PATH'] = '/tmp/finto:' + os.environ['PATH']   # ora "claude" è quello finto

## 1. `gate.sh`: the review as a quality gate

In [ ]:
%%bash
# a mini repository with a "feature" branch that changes a file
rm -rf /tmp/lab/repo && mkdir -p /tmp/lab/repo && cd /tmp/lab/repo
git init -q -b main . && git config user.email demo@example.com && git config user.name demo
printf 'def divide(a, b):\n    return a / b\n' > app.py
git add . && git commit -qm base
git checkout -qb feature
printf '\n# change\n' >> app.py && git commit -qam change
git log --oneline

In [ ]:
%%writefile /tmp/lab/gate.sh
#!/usr/bin/env bash
# modalità prudente di bash: fermati al primo errore, vietato usare
# variabili mai definite, niente errori nascosti nelle pipe
set -euo pipefail

# il branch di confronto: il primo argomento dello script, oppure
# "main" se non ne passi nessuno (è il significato di ${1:-main})
BASE="${1:-main}"

# le differenze del tuo lavoro rispetto al punto in cui ti sei staccato
# da BASE (la notazione "..." di git diff). Le catturiamo in una
# variabile: le passeremo a Claude via stdin, così non gli serve
# nessun permesso per leggerle.
DIFF=$(git diff "$BASE"...HEAD)

# niente differenze, niente review: usciamo con 0 ("tutto bene")
if [ -z "$DIFF" ]; then
  echo "Nessuna differenza rispetto a $BASE."
  exit 0
fi

# Il cuore dello script: la diff entra in Claude dalla pipe, e
# --json-schema obbliga la risposta ad avere ESATTAMENTE la forma
# descritta qui sotto — un oggetto con una lista "bloccanti", dove
# ogni voce ha file, riga, problema e scenario (tutti obbligatori:
# "required"). Cosi il verdetto e un dato che lo script puo leggere,
# non prosa da interpretare.
RISPOSTA=$(printf '%s' "$DIFF" | claude -p \
  "Review this diff. Report only defects that affect correctness or security and that have a concrete failure scenario. Ignore style, naming and formatting." \
  --output-format json \
  --json-schema '{
    "type": "object",
    "properties": {
      "bloccanti": {
        "type": "array",
        "items": {
          "type": "object",
          "properties": {
            "file":     { "type": "string" },
            "riga":     { "type": "integer" },
            "problema": { "type": "string" },
            "scenario": { "type": "string" }
          },
          "required": ["file", "riga", "problema", "scenario"]
        }
      }
    },
    "required": ["bloccanti"]
  }' \
  --max-turns 10) || {
    # Claude non è riuscito a girare: non è un fallimento della review.
    echo "gate: claude non ha completato il run" >&2
    exit 2
  }

# Con --json-schema, l'output validato non è tutta la risposta: vive nel campo
# structured_output dell'involucro JSON (i dettagli nel modulo 13). Da lì
# leggiamo la lista "bloccanti" definita dallo schema qui sopra.
N=$(printf '%s' "$RISPOSTA" | jq '.structured_output.bloccanti | length')

if [ "$N" -eq 0 ]; then
  echo "✔ Nessun problema bloccante."
  exit 0
fi

echo "✘ $N problemi bloccanti:"
printf '%s' "$RISPOSTA" | jq -r '
  .structured_output.bloccanti[]
  | "  \(.file):\(.riga)  \(.problema)\n      → \(.scenario)"
'
exit 1

In [ ]:
%%bash
cd /tmp/lab/repo && chmod +x /tmp/lab/gate.sh
echo '--- 1) the fake claude finds a problem ---'
/tmp/lab/gate.sh main 2>&1; echo "exit: $?"
echo
echo '--- 2) claude cannot run ---'
CLAUDE_FINTO_FALLISCE=1 /tmp/lab/gate.sh main 2>&1; echo "exit: $?"
echo
echo '--- 3) no difference to verify ---'
git checkout -q main
/tmp/lab/gate.sh main 2>&1; echo "exit: $?"

Three outcomes, three different exit codes: **1** = the review found problems, **2** = the tool did not work, **0** = nothing to report. A CI must be able to tell them apart.

## 2. `migra.sh`: batch migration, resumable

In [ ]:
%%writefile /tmp/lab/migra.sh
#!/usr/bin/env bash
# Nota: manca apposta la "-e" (esci al primo errore): un file che
# fallisce NON deve fermare la migrazione degli altri — i fallimenti
# li gestiamo noi, marcandoli e proseguendo.
set -uo pipefail

# argomenti con default: la lista dei file e quante migrazioni in parallelo
LISTA="${1:-files.txt}"
PARALLELI="${2:-4}"
# tre cartelle di lavoro in un colpo (la brace expansion {a,b,c} le crea tutte)
mkdir -p .migrazione/{ok,fail,log}

# la funzione che migra UN file
migra_uno() {
  local file="$1"
  local slug
  # trasforma il percorso in un nome di file valido: src/app.py -> src_app.py
  slug=$(printf '%s' "$file" | tr '/' '_')

  # IDEMPOTENZA (= rieseguire non fa danni): se questo file risulta gia
  # migrato da un run precedente, saltalo. E cio che rende lo script
  # ri-lanciabile dopo un'interruzione senza rifare tutto.
  [ -f ".migrazione/ok/$slug" ] && { echo "skip  $file"; return 0; }

  # l'invocazione vera: prompt preciso, permessi minimi, tetto ai turni,
  # e TUTTO l'output in un log dedicato a questo file (2>&1 = anche gli errori)
  if claude -p "Migrate $file from the legacy config API to the new one. \
Follow the pattern in src/config/new-api.ts. Do not change behaviour. \
Run \`npm run typecheck\` when done. Reply with exactly OK or FAIL." \
      --allowedTools "Read,Edit,Bash(npm run typecheck)" \
      --max-turns 15 \
      > ".migrazione/log/$slug.log" 2>&1
  then
    # Claude ha risposto: ha detto proprio "OK"?
    if grep -q '^OK' ".migrazione/log/$slug.log"; then
      touch ".migrazione/ok/$slug"; echo "ok    $file"; return 0
    fi
  fi

  # qualsiasi altro esito: marca il fallimento (il log dira perche)
  echo "$file" > ".migrazione/fail/$slug"; echo "FAIL  $file"; return 1
}
# rende la funzione visibile ai processi figli che xargs lancera
export -f migra_uno

# il motore del parallelismo: xargs legge la lista (da "< $LISTA") e lancia fino a
# PARALLELI processi insieme (-P), chiamando migra_uno su ogni file.
# La coda "bash -c '...' _ {}" e un idioma fisso: serve solo a passare
# il nome del file alla funzione — copialo cosi com'e.
xargs -P "$PARALLELI" -I{} bash -c 'migra_uno "$@"' _ {} < "$LISTA"

# riepilogo: conta i marcatori nelle due cartelle
echo
echo "riusciti: $(ls .migrazione/ok | wc -l)   falliti: $(ls .migrazione/fail | wc -l)"
echo "per ritentare i falliti:"
# ogni marcatore di fallimento contiene il percorso del file: li unisce in una nuova lista
echo "  cat .migrazione/fail/* > retry.txt && rm -rf .migrazione/fail && ./migra.sh retry.txt"

In [ ]:
%%bash
rm -rf /tmp/lab/batch && mkdir -p /tmp/lab/batch/src && cd /tmp/lab/batch
# quattro file da "migrare"; quello con "rotto" nel nome fa fallire il finto claude
printf 'src/app.py\nsrc/my_utils.py\nsrc/rotto_config.py\nsrc/db.py\n' > files.txt
chmod +x /tmp/lab/migra.sh
/tmp/lab/migra.sh files.txt 2

Three files succeeded (`ok`) and one failed (`FAIL`). Note that `src/my_utils.py` has an underscore in its name: the retry list rebuilds it correctly because each failure marker holds the real path.

In [ ]:
%%bash
cd /tmp/lab/batch
echo '--- second round: files already done are skipped (idempotence) ---'
/tmp/lab/migra.sh files.txt 2
echo
echo '--- retry list ---'
cat .migrazione/fail/* | tee retry.txt

In the second round you see `skip` for the three successful files and `FAIL` only for the broken one. `retry.txt` holds only the failed path, ready for `./migra.sh retry.txt`.